# Street-level features — “which signals actually matter?”

I already saved `cleanedGambling.csv`. For the **in-game AI** (the thing that shows win % on flop/turn/river), the code doesn’t read pre-baked columns out of that CSV for card math — it rebuilds the same vector **`build_stage_feature_payload`** produces in `scripts/features/poker_hand_strength.py`, and **`model_train.py`** does the exact same thing row-by-row when training.

So this notebook is me asking: *if I train a quick RandomForest on the same feature list as production (`STAGE_FEATURES`), what does the model lean on?* That’s not the same as causal truth, but it’s the kind of sanity check you do before trusting importances in a paper or a demo.

**You’ll need:** notebook 01 finished so `data/cleanedGambling.csv` exists.

## Setup + subsample (full training is `python model_train.py`)
Iterating on ~100k rows in a notebook is slow. I sample **whole `hand_id`s** so I don’t slice one street off another weirdly.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from model_train import build_training_frame
from scripts.models.feature_contracts import (
    DROPPED_FROM_STAGE_PAYLOAD_KEYS,
    STAGE_FEATURES,
)

sns.set_theme(style="whitegrid")

DATA_PATH = ROOT / "data" / "cleanedGambling.csv"
full = pd.read_csv(DATA_PATH)
nh = min(8000, full["hand_id"].nunique())
hands = full["hand_id"].drop_duplicates().sample(nh, random_state=42)
df = full[full["hand_id"].isin(hands)].copy()
print(f"full {full.shape} → notebook sample {df.shape} ({nh} hands)")

## What I removed from the **trained** stage models (not just plots)

When I sorted importances and stared at the correlation matrix, three payload keys kept landing at the bottom or looked redundant with stuff already in the vector:

- **`hero_stack_percentile`** — basically telling the same story as `hero_stack_bb` / `effective_stack_bb` / `spr`.
- **`is_short_stack`** — a coarse bucket the forest can approximate from continuous BB-normalized stacks.
- **`board_connected`** — overlaps how paired boards + straight draws already summarize connectivity.

Those keys are still *computed* inside `build_stage_feature_payload` for debugging, but **`model_train.py` / `poker_models.pkl` no longer consume them** — see `DROPPED_FROM_STAGE_PAYLOAD_KEYS` in `scripts/models/feature_contracts.py`. After pulling this change, run `python model_train.py` so `feature_names.pkl` stays in sync.

In [ ]:
print("Street keys removed from classifier inputs:", DROPPED_FROM_STAGE_PAYLOAD_KEYS)
for st, cols in STAGE_FEATURES.items():
    print(f"{st}: {len(cols)} features trained")

## Feature importance (preflop vs flop)
`model_train.py` wraps a RandomForest in **`CalibratedClassifierCV`** for probability quality; here I skip calibration just to eyeball **relative** importance faster.

**Preflop** uses mostly hole-card summaries + stack metrics. **Flop+** adds board texture / draws / shared-board danger — you’d expect board risk features to creep up after the flop.

In [ ]:
def importance_plot(stage: str):
    frame = build_training_frame(df, stage)
    cols = STAGE_FEATURES[stage]
    X = frame[cols].fillna(0.0)
    y = frame["won_flag"].astype(int)
    rf = RandomForestClassifier(
        n_estimators=200, random_state=42, max_depth=12, min_samples_leaf=5
    )
    rf.fit(X, y)
    imp = (
        pd.DataFrame({"feature": cols, "importance": rf.feature_importances_})
        .sort_values("importance")
    )
    fig_h = max(4.5, len(cols) * 0.2)
    fig, ax = plt.subplots(figsize=(9, fig_h))
    sns.barplot(data=imp, y="feature", x="importance", ax=ax, color="#486581")
    ax.set_title(f"RandomForest importance — stage={stage!r} (notebook diagnostic)")
    plt.tight_layout()
    plt.show()
    print("Top 5:")
    print(imp.tail(5).to_string(index=False))


importance_plot("preflop")

In [ ]:
importance_plot("flop")

## Correlation heatmap — “am I feeding redundant stuff?”
RandomForest can live with collinearity more than linear models, but if everything’s correlated you’ll see mushy importances. River stage has a fat feature list — quick visual.

In [ ]:
stage = "river"
frame = build_training_frame(df, stage)
cols = STAGE_FEATURES[stage]
X = frame[cols].fillna(0.0)
corr = X.corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax, square=False)
ax.set_title("Correlation matrix — river stage inputs")
plt.tight_layout()
plt.show()

## PCA — cheap 2D picture
Not replacing modeling with PCA — just checking whether wins/losses separate at all in this engineered space on the subsample. Huge overlap is normal (poker’s noisy).

In [ ]:
stage = "flop"
frame = build_training_frame(df, stage)
cols = STAGE_FEATURES[stage]
Xz = StandardScaler().fit_transform(frame[cols].fillna(0.0))
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(Xz)
scatter = pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1], "won": frame["won_flag"]})
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=scatter, x="PC1", y="PC2", hue="won", alpha=0.35, ax=ax)
ax.set_title(
    f"PCA on flop features — first 2 PCs explain {pca.explained_variance_ratio_.sum():.1%} var"
)
plt.tight_layout()
plt.show()
pd.Series(pca.explained_variance_ratio_, index=["PC1", "PC2"])

### What I take away
- Importance plots are **directional** — calibration + proper CV happens in `model_train.py`.
- The game’s inference path is literally: parse state → `build_stage_feature_payload` → `predict_stage_win_probability` (`stage_win_predictor.py`). I’m probing that contract.

**Next:** observable bluff model in **`03_visible_bluff_model.ipynb`** + run `python visible_bluff_train.py` for the calibrated pickle the UI reads.